In [ ]:
import importlib
for pkg in ["numpy", "matplotlib", "pypulseq", "MRzeroCore"]:
    importlib.import_module(pkg)
print("All libraries are installed.")

In [ ]:
from helpers import *
from visualizations import *

# Lab 5B: Writing Pulse Sequences

Howdy! Welcome to part two of dry lab five. We previously the effects of chemical shift / inhomogeneous $B_0$ on our spins, and I left ya wondering: is there a way to manually force the spins to _rephase_? In this lab, we'll do exactly this with the __spin echo__.

This lab is also our chance to practice some "real" MRI software to program __pulse sequences__; these are the timelines spelling out when exactly to play each RF pulse, when to turn on the gradients, and when to tell the ADC to start recording signal. In this lab, we'll use these tools:
+ [PyPulseq](https://github.com/imr-framework/pypulseq) is an open source package which lets us _write_ a pulse sequence in Python.
+ [MRZero](https://github.com/MRsources/MRzero-Core) lets us _run_ that pulse sequence on a simulated object, so we don't need a scanner to test it.

Let's get started...

## Part 0: Anatomy of a Pulse Sequence

In PyPulseq, a pulse sequence is a list of __blocks__, played one after another. Each block holds
+ an RF pulse,
+ gradients in $x$, $y$, and $z$,
+ and an ADC event (when is the receiver recording samples).

Everything inside one block plays _at the same time_, and the next block starts as soon as everything in the current one finishes. We make a pulse sequence by defining blocks and sticking them together.

Here are the PyPulseq functions we'll use in this lab.

| function                                      | the function's function                                                            |
|-----------------------------------------------|------------------------------------------------------------------------------------|
| `pp.make_block_pulse(flip_angle, ...)`        | a rectangular RF pulse, which tips spins by `flip_angle` radians                   |
| `pp.make_sinc_pulse(flip_angle, ...)`         | a sinc RF pulse for nicer slice selection                                          |
| `pp.make_trapezoid(channel, area=..., ...)`   | a gradient in `'x'`, `'y'`, or `'z'`. recall area $\propto$ $k$-space coordinate   |
| `pp.make_adc(num_samples, duration=..., ...)` | an ADC event that records `num_samples` samples over `duration` seconds            |
| `seq.add_block(...)`                          | adds one block, containing whatever events you pass it, to the end of the sequence |

Real scanner hardware limits our pulse sequence design choices; gradients have a max amplitude / ramp-up time (slew rate), and the ADC / RF-coils each need a little dead time before and after turning on. We tell PyPulseq these limits once with `pp.Opts`:

In [ ]:
system = pp.Opts(
    max_grad=40, grad_unit='mT/m',     # strongest gradient
    max_slew=170, slew_unit='T/m/s',   # fastest gradient ramp
    rf_dead_time=100e-6,               # wait before an RF pulse (s)
    rf_ringdown_time=20e-6,            # wait after an RF pulse (s)
    adc_dead_time=20e-6,               # wait before the ADC (s)
    B0=1.5,                            # main field (T)
)

## Part 1: The FID, Perhaps Review

The simplest pulse sequence is: $90^\circ$ excitation pulse $\rightarrow$ ADC on. We call the signal recorded __free induction decay__ (FID) because the spins are tipped into the transverse plane and then left alone ("free") while we record the voltage they induce in the RF coil. Write this sequence.

In [ ]:
seq_fid = pp.Sequence(system)
rf_dur = 1e-3     # s
adc_dur = 40e-3   # s
n_samples = 200

# TODO: make a 90 degree block pulse.
# HINT: angles in PyPulseq are in radians.
rf_90 = # ____
# TODO: make an ADC that records n_samples samples over adc_dur seconds.
adc = # ____

# TODO: add the two blocks to seq_fid: the RF pulse first, then the ADC.
seq_fid.add_block(# ____)
seq_fid.add_block(# ____)

print('timing ok:', seq_fid.check_timing()[0])

In [ ]:
# plot the pulse sequence. you should confirm that it looks right
plot_sequence(seq_fid)

Let's now simulate this pulse sequence. We'll use a phantom of $12$ spins with off-resonant frequencies spread evenly between $-80$ and $+80\,\text{Hz}$. Ask yourself: why are we making $T_2$ super long?

_Fun Facts: `simulate_seq` simulates our sequence with MRZero and returns the signal recorded at every ADC sample. `adc_times` returns when each of those samples was taken, and we will set $t=0$ when the center of the $90^\circ$ pulse occurs._

In [ ]:
dfs = np.linspace(-80, 80, 12)   # off-resonance of each isochromat, Hz
T2 = 100.0                       # s; basically no T2 decay

s_fid, _ = simulate_seq(seq_fid, offres_phantom(dfs, T2))
t_fid = adc_times(seq_fid) - center_time(rf_90)
plot_signal(t_fid, s_fid, title='FID')

This is the _total_ signal summed accross all the spins. But recall, if the object contains just _one_ spin, then the signal is exactly that spin's transverse magnetization. `simulate_isochromats` runs a simulation per isochromat, so we can watch each (just like in Lab 5). The black arrow is their average.


In [ ]:
mxy_fid = simulate_isochromats(seq_fid, dfs, T2)
animate_transverse_plane(mxy_fid, t_fid, every=2, title='FID')

Describe what you see. Why does the signal decay, even though each arrow stays the same length? Does this match what you saw in Lab 5?

<span style="color:cyan">
Your answer here.
</span>

## Part 2: The Spin Echo

In spin echo, we do a $90^\circ$ excite, let the spins dephase due to off-resonance, then do a $180^\circ$ to perfectly negate their phase. The spins who were ahead because they spun fast are now behind, and because they spin faster, they start to catch up. The slow spins are less behind, so they have less distance to make up, which is great because they're slow! After some time, each spin perfectly "makes up" the distance by which they got "ahead," and so there is no "phase dispersion" and the signal spikes again. This is called an *echo*. The time at which is echo occurs, we will refer to as $TE$.

We want our spins to realign at $TE$. **How long we should wait between the $90^\circ$ and $180^\circ$?**.


<span style="color:cyan">
Your answer here.
</span>

Consider the following analogy (likely review from in class):

BANG! The starting pistol is fired, and two racers speed off: a tortoise and a hare. This is no ordinary race, however! Seeing as the race-track-construction-companies are falling behind (due to inflation, "honey bucket" misconduct, and allegations of worker abuse), the track is only half-finished. To overcome this difficulty, the race organizers decide to have the runners *come back* after some time, or else race on unsafe track (they'd trip on potholes / trudge through mud / dodge bannana peels / leap over overgrown shrubs). Some time passes and before long, the hare disappears (hidden by the horizon). Meanwhile, the tortoise is taking scarcely his $66^{\text{th}}$ step... ...BANG! The come-back pistol is fired and the racers skid to a stop, turn on their heels, and race back at their same speed.

**Who wins? How's this different from spin echo, but a useful analogy nonetheless? How might you modify this analogy to more faithfully align with spin echo?**

<span style="color:cyan">
Your answer here.
</span>

The trickiest thing about writing a pulse sequence is always timing. Each pulse and ADC takes time (duh), and it's their _centers_ that need to be $T_E/2$ apart. Write a helper function that handles this: `fill_delay(first, second, gap)` should return the delay block to put between two events so that their centers end up `gap` seconds apart.

In [ ]:
def fill_delay(first, second, gap):
    """ returns a delay block such that the centers between blocks
        ~first~ and ~second~ are ~gap~ apart. note that a simple
        delay block of duration ~gap~ is insufficient, because this
        would delay the END of ~first~ and the START of ~second~
        by ~gap~.
    """

    # HINT: provided helpers center_time(block) and delay_block(duration)
    # could be useful. try help(function) to see what these do...
    after_first = # ____
    pass

Now write the spin echo. While the echo is _teeeeeechnically_ an instant in time, we should record also the "wind-up" and "wind-down" of the echo, since these periods have more signal than you'd see when the spins are totally dephased. Hence, you should define an ADC _centered at $T_E$_. The three building blocks (RF pulses and ADC) are done for you.


In [ ]:
seq_se = pp.Sequence(system)
TE = 20e-3        # s
adc_dur = 16e-3   # s
n_samples = 128

rf_90 = pp.make_block_pulse(np.pi / 2, duration=rf_dur, delay=system.rf_dead_time,
                            use='excitation', system=system)

adc = pp.make_adc(n_samples, duration=adc_dur, delay=system.adc_dead_time, system=system)

rf_180 = pp.make_block_pulse(np.pi, duration=rf_dur, delay=system.rf_dead_time,
                             use='refocusing', phase_offset=np.pi / 2, system=system)

# TODO: add five blocks: 90 -> appropriate delay -> 180 -> appropriate delay -> ADC.
# HINT: use fill_delay for your delay blocks!
seq_se.add_block(rf_90)
seq_se.add_block(# ____)
seq_se.add_block(# ____)
seq_se.add_block(# ____)
seq_se.add_block(# ____)

print('timing ok:', seq_se.check_timing()[0])
plot_sequence(seq_se)

In [ ]:
# -------- same simulations as done for the FID --------

s_se, _ = simulate_seq(seq_se, offres_phantom(dfs, T2))
t_se = adc_times(seq_se) - center_time(rf_90)
plot_signal(t_se, s_se, TE=TE, title='spin echo')

In [ ]:
mxy_se = simulate_isochromats(seq_se, dfs, T2)
animate_transverse_plane(mxy_se, t_se, every=2, title='spin echo')

What happens to the echo if you change $T_E$? What happens if you play the $180^\circ$ pulse early, by using a gap of (say) `0.3 * TE` instead of `0.5 * TE` for the first delay? Answer these questions; then go back and re-run the simulations to confirm or deny; and then revise your answers. Note that you will have to keep `adc_dur` a bit shorter than $T_E$ (why?).

<span style="color:cyan">
Your answer here.
</span>

### Reconsidering $T_2$

So far, we have been running simulations where $T_2$ is huge. In reality, $T_2$ is often tens of milliseconds, which is on the order of $T_E$, and therefore non-negligible. Re-run the spin echo with $T_2 = 20\,\text{ms}$, and compare the height of the echo to the one with a long $T_2$.

In [ ]:
T2

In [ ]:
T2_long = # ____
T2_short = # ____

s_se_short, _ = simulate_seq(seq_se, offres_phantom(dfs, T2_short))
s_se_long, _ = simulate_seq(seq_se, offres_phantom(dfs, T2_long))

plot_signal(t_se, [s_se, s_se_short], names=['long $T_2$', '$T_2$ = 20 ms'], TE=TE,
            title='spin echo with and without $T_2$ decay')
i_TE = np.argmin(np.abs(t_se - TE))

In [ ]:
print(f'echo height ratio:  {np.abs(s_se_short[i_TE]) / np.abs(s_se[i_TE]):.2f}')
print(f'exp(-TE / T2):      {np.exp(-TE / T2_short):.2f}')

The echo still forms, but it is smaller by a factor of $e^{-T_E/T_2}$, as the print statements above confirm. Why is it true? Why can the $180^\circ$ pulse undo the dephasing from off-resonance, but not $T_2$ decay? Why is it not possible to undo $T_2$ decay, even with a different strategy? So, why might someone choose a spin echo over a plain FID, if $T_2$ decay is inevitable?

<span style="color:cyan">
Your answer here.
</span>

## Part 3: A $2\text{D}$ Gradient Echo Image

Time to make an image! Recall that we fill $2\text{D}$ $k$-space one line at a time. We call each repetition a TR, and every TR does the following
1. excite
2. blip $G_y$ to pick a $k_y$ line (_phase encode_), and prewind $k_x$ to $-k_{\max}$
3. play $G_x$ and turn on the ADC (_frequency encode_)
4. wait for the rest of the TR, so the spins can recover before repeating

This is called a __gradient echo__ (GRE): there is no $180^\circ$ pulse; the "echo" is just the moment the readout gradient passes through $k_x = 0$. First, let's make each piece. Recall that the area beneath a gradient pulse is how far it moves us in $k$-space, and that each step in $k$ should be $\Delta k = 1/\text{FOV}$.

In [ ]:
# -------- system / scan parameters --------
gre = pp.Opts(
    max_grad=20, grad_unit='mT/m', max_slew=100, slew_unit='T/m/s',
    rf_dead_time=100e-6, rf_ringdown_time=20e-6, adc_dead_time=20e-6, B0=1.5,
)
N = 64            # image is N by N
fov = 256e-3      # m
dk = 1 / fov      # 1/m
TE = 5e-3         # s
TR = 100e-3       # s
flip_deg = 30

# -------- excitation --------
# pp.make_sinc_pulse returns the rf excite as well as the
# gradient gz (& gz_reph to rewind phase dispersion)
rf, gz, gz_reph = pp.make_sinc_pulse(
    flip_deg * np.pi / 180, duration=2e-3, slice_thickness=5e-3, apodization=0.5,
    time_bw_product=4, system=gre, use='excitation', return_gz=True,
    delay=gre.rf_dead_time,
)

# -------- readout --------
# pp.make_trapezoid returns a trapezoidal (so as to satisfy slew rate)
# pulse with a flat region that brings walks across k_x while the ADC records N samples (lab 4!)
gx = pp.make_trapezoid('x', flat_area=N * dk, flat_time=3.2e-3, system=gre)
adc = pp.make_adc(N, duration=gx.flat_time, delay=gx.rise_time, system=gre)

# TODO: the prewinder, which moves k_x from 0 to -k_max before the readout.
# HINT: gx.area is the readout's area, which is the readout's total k-space movement.
gx_pre = # ____

# TODO: the area of the phase-encode gradient for each of the N lines.
# HINT: this is an array [-(N//2) * dk, ..., 0, ..., (N//2 - 1) * dk].
#       use np.arange(N) to get an array [0, 1, ..., N - 1, N].
phase_areas = # ____

# -------- timing ---------
# center of RF pulse -> center of ADC
# desired TE - time filled by slice selection - time filled by
delay_te = (TE - (pp.calc_duration(rf, gz) - center_time(rf))
               - pp.calc_duration(gx_pre, gz_reph) - center_time(adc))
# whatever is left of the TR, after the other four blocks
# this is given, to permit the spins time to relax
delay_tr = TR - pp.calc_duration(rf, gz) - pp.calc_duration(gx_pre, gz_reph) - delay_te - pp.calc_duration(gx)

Now, write the loop. Each time through the loop is one TR, and adds the five blocks above. Remember that events in the same block play at the same time!

In [ ]:
seq_gre = pp.Sequence(gre)

for i in range(N):
    # TODO: the phase-encode gradient for line i, on the 'y' channel.
    # HINT: pp.make_trapezoid(channel, area=..., duration=1e-3, system=gre)
    gy_pre = # ____

    # TODO: add the five blocks. add_block(a, b) adds both blocks
    #       to the sequence, such that they will play simultaneously!
    seq_gre.add_block(# ____)  # hint ~ play RF & appropriate z gradient
    seq_gre.add_block(# ____)  # hint ~ play all of the prewinding / rephasing gradients
    seq_gre.add_block(# ____)  # hint ~ choose the appropriate delay block
    seq_gre.add_block(# ____)  # hint ~ play the readout gradient and turn on the ADC
    seq_gre.add_block(# ____)  # hint ~ choose the appropriate delay block

print('timing ok:', seq_gre.check_timing()[0])
print(f'scan time: {seq_gre.duration()[0]:.1f} s')

The whole sequence is $64$ TRs, which is too many to look at. Here are the first two, only. Try to recognize all the blocks!

In [ ]:
plot_sequence(seq_gre, time_range=(0, 2 * TR))

Let's simulate! MRZero hands back one long list of ADC samples: all $N$ samples from the first line, then all $N$ from the second, and so on. It also gives us the $k$-space location of every sample, so we can check that we really did sample a Cartesian grid. Make sure you have a vague idea of what each line of the code below does; then run it and check out the result.

In [ ]:
signal, ktraj = simulate_seq(seq_gre)
plot_kspace_samples(ktraj)

kspace = signal.reshape(N, N)
img_gre = centered_ifft(kspace)

show_images([np.log(1e-3 + np.abs(kspace)), img_gre], ['log |k-space|', 'gradient echo'])

Look at the scan time we printed above - $6$ seconds is quite long for a single $64 \times 64$ slice. Real images are usually $256$ by $256$ or more, and we usually want to record several slices. With our TR, how long would a $256$ by $256$ scan take? *HINT: how many $TR$s are does it take to run the full sequence?*

<span style="color:cyan">
Your answer here.
</span>

## Part 4: Spoiling (Optional / Read Only)

Let's try to speed up our scan by shortening the TR from $100\,\text{ms}$ to $12\,\text{ms}$. `make_gre` builds the exact same sequence you just wrote$^*$ assuming you and I did it the same way. Run the next cell, and compare the two images.

$^*$rather, claude (not into writing that much code myself) did it

In [ ]:
seq_fast = make_gre(gre, TE=5e-3, TR=12e-3, flip_deg=30, spoil=False)
signal_fast, _ = simulate_seq(seq_fast)
img_fast = centered_ifft(signal_fast.reshape(N, N))

show_images([img_gre, img_fast], ['TR = 100 ms', 'TR = 12 ms'])

Yikes! What happened?

Our sequence assumes that each TR starts fresh: the spins along $z$, with nothing in the transverse plane. With a short TR that isn't true anymore. Tissues with a long $T_2$ still have some transverse magnetization left over when the next RF pulse plays, and it carries the phase from the _previous_ line of $k$-space$^*$.

There are three tools we might use to solve this problem (we have so far used only the first):
1. __Wait.__ A long TR permits the leftover signal time to decay on its own.
2. __Gradient spoiling.__ A huge gradient at the end of each TR applies a ton of phase dispersion, such that the spins within each voxel cancel (just like our FID!).
3. __RF spoiling.__ Change the phase of the RF pulse from TR to TR, in a carefully chosen pattern such that leftover signal from different TRs doesn't interfere.

`make_gre(..., spoil=True)` turns on gradient spoiling and RF spoiling. It also plays some __dummy TRs__ (RF pulses and gradients, but no ADC) before the first real line. This is because there is less signal after the first few TRs, because the spins are initially fully recovered in $z$, but with short $TR$, that is not true in the steady state. We play a few dummy TRs so that all the signal comes from excitations with similar starting magnitudes.

Here are two TRs of the spoiled sequence, right after the dummies. Compare them to your sequence diagram from Part 3 and locate spoilers.

$^*$ you may think, also, that there was not as much time for $T_1$ recovery. this is true! but it turns out it's *not* generally a concern, because it won't really mess up the signal we recover at all. while we might attenuate slightly some of the signal by choosing a much shorter $TR$, we are mostly concerned with interference from surviving spins in the transverse plane, corrupting the signal we *do* record.

In [ ]:
seq_spoiled = make_gre(gre, TE=5e-3, TR=12e-3, flip_deg=30, spoil=True)
n_dummy = 100
plot_sequence(seq_spoiled, time_range=(n_dummy * 12e-3, (n_dummy + 2) * 12e-3))

In [ ]:
# ------- run the experiment and compare results --------
signal_spoiled, _ = simulate_seq(seq_spoiled)
img_spoiled = centered_ifft(signal_spoiled.reshape(N, N))
show_images([img_gre, img_fast, img_spoiled],
            ['TR = 100 ms', 'TR = 12 ms', 'TR = 12 ms, spoiled'])

Notice that the spoiled image has much stronger contrast: white matter is brighter than the gray matter, and the fluid in the middle of the brain (CSF) is darker. with only $12\,\text{ms}$ between pulses, tissues don't have as much time to grow back along $z$, which emphasizes differences in $T_1$. Hence, the ones who's magnetization regrows fastest (namely, white matter) give a lot more signal relative to those with short $T_1$.

## Part 5: Contrast (Optional / Light Coding)

Why does an MRI scan show anatomy at all? Different tissues have different proton density ($PD$), $T_1$, and $T_2$. If we choose the sequence timing well, we can make an image whose brightness mostly depends on _one_ of those. Here are the maps for MRZero's brain. Which tissue do you think is which?

In [ ]:
PD, T1, T2 = brain_maps()
show_images([PD, T1, T2], ['proton density', '$T_1$', '$T_2$'])

In each TR of a spoiled gradient echo, we have two main knobs to twist, to determine the appearance of different tissues:
+ __TR controls $T_1$ weighting.__ Each RF pulse tips some magnetization away from $z$, and the spins regrow along $z$ with time constant $T_1$ before the next pulse. With a short TR, only tissues with a _short_ $T_1$ regrow enough to give much signal. With a long TR, every tissue has time enough to fully recover, so $T_1$ doesn't matter.
+ __TE controls $T_2^*$ weighting.__ After the pulse, the signal decays with time constant $T_2^*$ until we record the echo at $T_E$. With a short TE, nothing has decayed yet, so $T_2^*$ doesn't matter. With a long TE, only tissues with a $T_2^*$ still have signal, so we can differentiate tissues by $T_2^*$.

If we make both effects small (i.e. long $TR$ so all tissues have full $T_1$ recover, and short $TE$ so no tissues have experience $T_2$ decay), the only thing left to determine difference in signal is the density of hydrogens, yielding a __proton-density__ weighted image.

Choose a TE, TR, and flip angle to make each of the three images below. Use $T_E$ between $5$ and $40\,\text{ms}$, TR between $15$ and $300\,\text{ms}$, and a flip angle between $10^\circ$ and $40^\circ$.

In [ ]:
# TODO: choose (TE in s, TR in s, flip angle in degrees) for each image.
# HINT: flip angle doesn't matter too much, but we can pick a larger
#       one to emphasize T1 differences. think about why!
TE_T1w, TR_T1w, flip_T1w = # ____
TE_PDw, TR_PDw, flip_PDw = # ____
TE_T2w, TR_T2w, flip_T2w = # ____

images = []
for TE_i, TR_i, flip_i in [(TE_T1w, TR_T1w, flip_T1w), (TE_PDw, TR_PDw, flip_PDw), (TE_T2w, TR_T2w, flip_T2w)]:
    seq_i = make_gre(gre, TE=TE_i, TR=TR_i, flip_deg=flip_i)
    signal_i, _ = simulate_seq(seq_i)
    images.append(centered_ifft(signal_i.reshape(N, N)))

show_images(images, ['$T_1$ weighted', 'PD weighted', '$T_2^*$ weighted'])

Congrats on finishing the final lab. You wrote an FID, a spin echo, and a $2\text{D}$ gradient echo with PyPulseq, ran them with MRZero, and saw how the timing of a sequence affects the final image. I hope it was fun! As always, please hit me up with any feedback.

___this lab and accompanying visualization written by Ryan Black for the MRI decal, fall 2026___